In [ ]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.datasets import make_classification 
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.feature_selection import SelectKBest, SelectFpr, chi2, mutual_info_classif, f_classif
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import SnowballStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
import re
import string
from bs4 import BeautifulSoup
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import confusion_matrix

import nltk
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [ ]:
rows = df.loc[df.duplicated(subset=['article', 'title'])]
duplicated = df.loc[df.duplicated(subset=['title','article'])]

df.dropna(inplace=True)

In [ ]:
def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    return text

for i, article in zip(df.index, df['article']):
    df.loc[i, 'article'] = clean_text(article)

In [ ]:
display(df)

In [ ]:
#mask = ((df['label'] == 0) | (df['label'] == 5))
#subset = df[mask]

In [ ]:
#df = subset

In [ ]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
text = df['title'] + ' ' + df['title'] + ' ' + df['article']
df = pd.concat([df, text], axis=1)

#rinomino le colonne e droppo title e article
df.rename(columns={0: 'text'}, inplace=True)
df.drop(['title', 'article'], inplace=True, axis=1)
df.columns

In [ ]:
stemmer = SnowballStemmer('english')
def stemming_tokenizer(text):
    # Separa le parole
    words = text.split()
    # Applica lo stemmer a ogni parola e riunisci
    # (È qui che avviene la magia: "running" -> "run")
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)

df['text'] = df['text'].apply(stemming_tokenizer)

In [ ]:
#y = df['label']
#df.drop('label', inplace=True, axis=1)

In [ ]:
df.drop('timestamp', inplace=True, axis=1)

In [ ]:
df.drop('Id', inplace=True, axis=1)

In [ ]:
df.drop('page_rank', inplace=True, axis=1)

In [ ]:
stop_words = list(ENGLISH_STOP_WORDS) + [
    # --- Rumore HTML e Scraping ---
    'http', 'https', 'src', 'href', 'img', 'alignleft', 'alignright', 
    'height', 'width', 'border', 'clearall', 'borderimga', 'br', 'class',
    
    # --- Agenzie Stampa e Formattazione News ---
    'reuters', 'ap', 'afp', 'pa', 'upi', 'press', 'association',
    'new', 'york', 'london', 'washington', # Città ricorrenti nelle dateline
    'said', 'says', 'told', 'reported', 'added', # Verbi di reportistica
    
    # --- Temporali Generici (inutili per classificare l'argomento) ---
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'yesterday', 'today', 'tomorrow', 'year', 'years', 'week', 'month', 'day',
    'time', 'daily', 'date', 'attack', 'baghdad', 'bomb', 'bush', 'gaza', 'iran', 'iraq',
    'iraqi', 'israel', 'kerry', 'palestinian','british' ,'countri', 'court','death', 'democrat' ,'elect' ,'forc',
 'govern' ,'group' ,'kill', 'leader', 'militari', 'minist', 'nation', 'offici',
 'peopl', 'plan', 'polic', 'presid', 'prime', 'prime minist', 'reuters', 'secur',
 'state', 'talk' ,'unit', 'vote', 'war' ,'world']

stemmed_stop_words = [stemmer.stem(word) for word in stop_words]

In [ ]:
df

In [ ]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [ ]:
encoder = OneHotEncoder(min_frequency=100, handle_unknown='infrequent_if_exist')

scaler = MinMaxScaler()

text_pipeline_selector = Pipeline([
    ('vect', TfidfVectorizer(
        stop_words=stemmed_stop_words,
        max_df = 0.8,
        max_features = 30000,
        ngram_range=(1, 2),
        use_idf=False
    )),
    ('selector', SelectFpr(score_func=chi2, alpha=0.001)),
    ('selector2', SelectFpr(score_func=f_classif, alpha=0.01))
])

vectorizer = TfidfVectorizer(
        max_features=30000,
        stop_words='english',
        min_df=3)

vectorizer_title = TfidfVectorizer(
        max_features=3000,
        stop_words=stemmed_stop_words,
        ngram_range=(1, 2),
        min_df=3)

preprocessor = ColumnTransformer(transformers=[
    ('text', vectorizer, 'text'),
    ('source', encoder, ['source'])
], remainder='drop')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y, shuffle=True)

In [ ]:
X_train_final = preprocessor.fit_transform(X_train, y_train)
X_test_final = preprocessor.transform(X_test)

In [ ]:
mask_train = (y_train == 0) | (y_train == 5)
mask_train

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

# Definiamo le classi difficili (nel tuo caso sembrano essere 0 e 5)
classi_difficili = [0, 5]

# Creiamo la maschera usando y_train (che è una Series pandas)
mask_train = y_train.isin(classi_difficili)

# IMPORTANTE: Tagliamo direttamente la matrice sparsa X_train_final
# Non convertirla in DataFrame!
X_train_bin = X_train_final[mask_train.values] # .values converte la serie in array numpy per lo slicing
y_train_bin = y_train[mask_train]

print(f"Dataset Binario creato. Shape: {X_train_bin.shape}")
# Definisci la SVM base
base_svm = LinearSVC(
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

# AVVOLGILO per avere le probabilità
# method='sigmoid' (Platt scaling) o 'isotonic'
clf_general = CalibratedClassifierCV(base_svm, method='sigmoid') 

# Il modello esperto (Binario) può rimanere LogisticRegression (che ha già predict_proba nativo)
# o una SVM calibrata anche lei.
clf_expert = RandomForestClassifier(
    n_estimators=1500,
    criterion='entropy',
    max_depth=20,
    class_weight='balanced'
)


# Fai il fit come prima
print("Addestramento Generale con Calibrazione...")
clf_general.fit(X_train_final, y_train)

print("Addestramento Esperto...")
clf_expert.fit(X_train_bin, y_train_bin)

In [ ]:
def predict_cascade_threshold(X, model_general, model_expert, difficult_classes, threshold=0.75):
    """
    threshold: Valore tra 0 e 1. 
               Se il modello generale è sicuro meno di questa soglia, chiediamo all'esperto.
               Es: 0.75 significa "Se sei sicuro meno del 75%, passo la palla".
    """
    
    # 1. Ottieni le classi predette
    preds = model_general.predict(X)
    
    # 2. Ottieni le probabilità (matrice N_samples x N_classes)
    probas = model_general.predict_proba(X)
    
    # Calcola la confidenza massima per ogni riga (quanto è sicuro della sua scelta?)
    confidence = np.max(probas, axis=1)
    
    # 3. Maschera 1: È una classe difficile?
    mask_difficult_class = np.isin(preds, difficult_classes)
    
    # 4. Maschera 2: Il modello è incerto? (confidenza < soglia)
    mask_uncertain = confidence < threshold
    
    # 5. Combinazione: Deve essere SIA una classe difficile SIA incerto
    mask_trigger_expert = mask_difficult_class & mask_uncertain
    
    # Se nessun campione soddisfa i criteri, ritorna subito
    if not np.any(mask_trigger_expert):
        return preds
        
    # 6. Isola i campioni e predici con l'esperto
    X_doubtful = X[mask_trigger_expert]
    expert_preds = model_expert.predict(X_doubtful)
    print(f'{mask_trigger_expert.sum()}')
    # 7. Sovrascrivi
    preds[mask_trigger_expert] = expert_preds
    
    return preds

In [ ]:
# Prova con una soglia del 70% (0.7)
y_pred_smart = predict_cascade_threshold(
    X_test_final, 
    clf_general, 
    clf_expert, 
    difficult_classes=[0, 5], 
    threshold=0.70
)

print("\n--- Report con Soglia di Incertezza ---")
print(classification_report(y_test, y_pred_smart))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

def plot_confusion_matrix_fatta_bene(y_true, y_pred, labels):
    """
    y_true: lista delle etichette reali
    y_pred: lista delle etichette predette dal modello
    labels: lista dei nomi delle classi (es. ['Sport', 'Politica', 'Esteri'])
    """

    # 1. Calcolo della matrice NORMALIZZATA
    # normalize='true' fa sì che la somma di ogni RIGA sia 1 (100%).
    # Questo ti dice: "Della classe X, quanti ne ho indovinati?"
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')

    # 2. Setup del grafico
    plt.figure(figsize=(10, 8)) # Dimensioni generose

    # 3. Disegno con Seaborn
    # annot=True: scrive i numeri nelle celle
    # fmt='.1%': formatta come percentuale con 1 decimale (es. 85.4%)
    # cmap='Blues': scala di blu (più scuro = più alto), ottima per report
    sns.heatmap(cm, 
                annot=True, 
                fmt='.1%', 
                xticklabels=labels, 
                yticklabels=labels, 
                cmap='Blues',
                cbar_kws={'label': 'Proporzione di predizioni'})

    # 4. Etichette assi (Cruciale per non confondersi)
    plt.ylabel('Realtà (True Label)', fontsize=12, fontweight='bold')
    plt.xlabel('Predizione (Predicted Label)', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix Normalizzata', fontsize=14)
    
    plt.tight_layout()
    plt.show()

# Esempio di utilizzo fittizio
# labels_list = ['Sport', 'Politica', 'Cronaca']
# plot_confusion_matrix_fatta_bene(y_test, predictions, labels_list)

plot_confusion_matrix_fatta_bene(y_test, y_pred_smart, labels=sorted(y_test.unique()))


In [ ]:
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import confusion_matrix

weights = compute_sample_weight(class_weight='balanced', y=y_train)


clf = LogisticRegression(
    C=1,                       
    class_weight='balanced',    
    solver='liblinear',        
    penalty='l2',           
    max_iter=1000,              
    random_state=42
)
clf2 = RandomForestClassifier(
    random_state= 42,
    n_estimators=1500,
    max_depth=10,
    class_weight='balanced',
    n_jobs = -1
)

clf3 = GradientBoostingClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    random_state=42,
    warm_start=True,
)

svm = LinearSVC(
        C=1,
        class_weight='balanced',
        max_iter=2000,
        random_state=42,
        penalty='l1'
)

clf5 = XGBClassifier(
        n_estimators=1000,      
        learning_rate=0.05,     
        max_depth=6,             
        min_child_weight=1,
        gamma=0.1,           
        subsample=0.8,           
        colsample_bytree=0.8,     
        n_jobs=-1,               
        random_state=42,
        eval_metric='mlogloss'  
    )

clf_svm_prob = CalibratedClassifierCV(svm, method='sigmoid')
clf_nb = MultinomialNB()

voting_clf = VotingClassifier(
    estimators=[ 
        ('svm', clf_svm_prob), 
        ('xgb', clf5)
    ],
    voting='soft', weights=[2,1], verbose=True
)

model = svm

model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))